# 🎓 বাংলা NCTB ও সাধারণ জ্ঞান ডেটাসেট দিয়ে SmolLM2-135M (100M Class) ফাইন-টিউনিং

এই নোটবুকটি দিয়ে আপনি **SmolLM2-135M** (১৩৫ মিলিয়ন প্যারামিটারের আসল আল্ট্রা-লাইট মডেল) ফাইন-টিউন করতে পারবেন।

**💡 কেন এই মডেল?**
- ৪-বিট GGUF ফাইল সাইজ হবে মাত্র **~৮৫ MB** (৩৮০ MB নয়!)।
- ফোনে চলার সময় পিক র‍্যাম খরচ হবে মাত্র **১৫০–১৮০ MB**।
- **২ GB RAM-এর itel, Symphony বা কমদামী ফোনে কোনো ল্যাগ বা ক্র্যাশ ছাড়া পানির মতো চলবে!**

---

**ধাপসমূহ**:
1. পরিবেশ সেটআপ (২ মিনিট)
2. ডেটাসেট লোড করা (১ মিনিট)
3. ১৩৫M মডেল ফাইন-টিউনিং (১৫-২০ মিনিট)
4. টেস্টিং ও GGUF-এর জন্য সেভ (২ মিনিট)


## 📌 ধাপ ১: GPU চেক করুন

In [ ]:
# GPU চেক করুন
!nvidia-smi

## 📁 ধাপ ২: Google Drive মাউন্ট করুন

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 📦 ধাপ ৩: প্রয়োজনীয় লাইব্রেরি ইনস্টল করুন

এই সেলটি চালাতে **৩-৫ মিনিট** সময় লাগবে।

In [ ]:
%%capture
# প্রয়োজনীয় প্যাকেজ ইনস্টল
!pip install -q transformers datasets accelerate peft bitsandbytes trl torch

## 🔧 ধাপ ৪: ডেটাসেট লোড ও প্রস্তুতি

**গুরুত্বপূর্ণ**: নিচের কোডে `DATASET_PATH` পরিবর্তন করে আপনার ডেটাসেটের সঠিক পাথ দিন।

In [ ]:
import json
from datasets import Dataset

# ✏️ আপনার ডেটাসেটের পাথ এখানে লিখুন
# নোট: এই রেপোতে ৪০৮টি ভ্যালিড NCTB প্রশ্ন-উত্তর আছে
DATASET_PATH = "/content/drive/MyDrive/nctb_dataset_500plus.jsonl"

# JSONL ফাইল লোড করুন (এরর হ্যান্ডলিং সহ)
data = []
error_lines = []

print("📂 ফাইল লোড হচ্ছে...\n")

with open(DATASET_PATH, 'r', encoding='utf-8') as f:
    for line_num, line in enumerate(f, 1):
        line = line.strip()
        if not line:  # খালি লাইন স্কিপ
            continue
        
        try:
            # JSON parse করুন
            item = json.loads(line)
            
            # Validate: instruction এবং output আছে কিনা চেক করুন
            if 'instruction' in item and 'output' in item:
                data.append(item)
            else:
                print(f"⚠️  লাইন {line_num}: 'instruction' বা 'output' ফিল্ড নেই, স্কিপ করা হলো")
                error_lines.append(line_num)
        
        except json.JSONDecodeError as e:
            print(f"❌ লাইন {line_num}: JSON এরর - {str(e)[:50]}... স্কিপ করা হলো")
            error_lines.append(line_num)
            continue

print(f"\n{'='*60}")
print(f"✅ সফলভাবে লোড হয়েছে: {len(data)} টি")

if error_lines:
    print(f"⚠️  সমস্যাযুক্ত লাইন: {len(error_lines)} টি (লাইন নম্বর: {error_lines[:10]}{'...' if len(error_lines) > 10 else ''})")
else:
    print(f"🎉 সব ডেটা পারফেক্ট!")

print(f"{'='*60}\n")

# প্রথম কয়েকটি উদাহরণ দেখুন
if data:
    print(f"📝 প্রথম ৩টি উদাহরণ:\n")
    for i, example in enumerate(data[:3], 1):
        print(f"{i}. প্রশ্ন: {example['instruction'][:80]}...")
        print(f"   উত্তর: {example['output'][:80]}...\n")
else:
    print("❌ কোনো ভ্যালিড ডেটা লোড হয়নি! ফাইল চেক করুন।")
    raise ValueError("ডেটাসেট খালি!")

# Dataset অবজেক্ট তৈরি
dataset = Dataset.from_list(data)

# Train/Test split (95% train, 5% test)
dataset = dataset.train_test_split(test_size=0.05, seed=42)

print(f"\n📊 ডেটাসেট বিভাজন:")
print(f"  - Training: {len(dataset['train'])} টি")
print(f"  - Testing: {len(dataset['test'])} টি")
print(f"\n✅ ডেটাসেট প্রস্তুত!")

## 🤖 ধাপ ৫: ১৩৫M (100M Class) আল্ট্রা-লাইট মডেল নির্বাচন

লো-র‍্যাম অ্যান্ড্রয়েড মোবাইল (২ GB RAM)-এর জন্য **HuggingFaceTB/SmolLM2-135M-Instruct** হলো পৃথিবীর অন্যতম সেরা ও দ্রুতগতির মডেল।

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import prepare_model_for_kbit_training

# ⭐ 100M Class Mobile-Friendly Model (২ জিবি মোবাইলে পানির মতো চলার জন্য)
MODEL_NAME = "HuggingFaceTB/SmolLM2-135M-Instruct"

print(f"📌 নির্বাচিত মডেল: {MODEL_NAME}")
print("📥 টোকেনাইজার ও মডেল ডাউনলোড হচ্ছে...\n")

# Tokenizer লোড
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# ৪-বিট কোয়ান্টাইজেশন কনফিগারেশন
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

# Model লোড
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
    torch_dtype=torch.bfloat16,
)

# ট্রেনিংয়ের জন্য প্রস্তুত
model = prepare_model_for_kbit_training(model)

print("\n✅ ১৩৫M মডেল সফলভাবে লোড হয়েছে!")
print(f"\n📊 মডেলের তথ্য:")
print(f"  - মডেল আইডি: {MODEL_NAME}")
print(f"  - আর্কিটেকচার: {model.config.model_type} (LLaMA-based)")
print(f"  - হিডেন সাইজ: {model.config.hidden_size}")
print(f"  - লেয়ার সংখ্যা: {model.config.num_hidden_layers}")
print(f"  - মেমোরি খরচ: {torch.cuda.memory_allocated() / 1e9:.2f} GB (খুবই কম!)")


## ⚙️ ধাপ ৬: LoRA কনফিগারেশন সেটআপ

**LoRA** (Low-Rank Adaptation) ব্যবহার করে আমরা শুধু মডেলের একটি ছোট অংশ ট্রেন করব।

In [ ]:
from peft import LoraConfig, get_peft_model

# LLaMA আর্কিটেকচারের জন্য অপ্টিমাইজড LoRA কনফিগারেশন
lora_config = LoraConfig(
    r=16,                      # LoRA rank
    lora_alpha=32,             # LoRA alpha
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

# মডেলে LoRA প্রয়োগ
model = get_peft_model(model, lora_config)

# Trainable প্যারামিটার তথ্য
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
all_params = sum(p.numel() for p in model.parameters())

print("✅ LoRA কনফিগ সফল!")
print(f"\n📊 প্যারামিটার তথ্য:")
print(f"  - মোট প্যারামিটার: {all_params:,} (~১৩৫ মিলিয়ন)")
print(f"  - ট্রেন করা হবে: {trainable_params:,} ({100 * trainable_params / all_params:.2f}%)")


## 📝 ধাপ ৭: ডেটাসেট ফরম্যাটিং (ChatML Template)

SmolLM2 মডেল ChatML ফরম্যাট সাপোর্ট করে (`<|im_start|>` ও `<|im_end|>` টোকেন)।

In [ ]:
def format_prompt(example):
    """
    বাংলা ডেটাসেটকে ChatML instruction ফরম্যাটে রূপান্তর
    """
    text = f"""<|im_start|>system
You are a helpful AI assistant for Bengali education and general conversation.<|im_end|>
<|im_start|>user
{example['instruction']}<|im_end|>
<|im_start|>assistant
{example['output']}<|im_end|>"""
    return {"text": text}

# ডেটাসেট ম্যাপ করুন
formatted_dataset = dataset.map(format_prompt, remove_columns=dataset['train'].column_names)

print("✅ ডেটাসেট ফরম্যাট সম্পন্ন!")
print(f"\n📄 ফরম্যাটেড স্যাম্পল:\n{formatted_dataset['train'][0]['text'][:250]}...")


## 🎯 ধাপ ৮: Training Arguments সেটআপ

এখানে আপনি training-এর বিভিন্ন প্যারামিটার কনফিগার করতে পারবেন।

In [ ]:
from trl import SFTConfig

# আউটপুট ডিরেক্টরি (Drive-এ সেভ হবে)
output_dir = "/content/drive/MyDrive/smollm_135m_finetuned"

# ১৩৫M মডেলের জন্য অপ্টিমাইজড ট্রেনিং কনফিগারেশন
training_args = SFTConfig(
    output_dir=output_dir,
    
    # ট্রেনিং পরিকল্পনা (১৩৫M মডেল Colab T4-এ সুপার ফাস্ট ট্রেন হবে!)
    max_steps=1500,                        # ১,৫০০ স্টেপ (~১৫-২০ মিনিট)
    num_train_epochs=2,
    per_device_train_batch_size=4,         # ১৩৫M হালকা হওয়ায় ব্যাচ সাইজ ৪ সম্পূর্ণ নিরাপদ
    gradient_accumulation_steps=4,         # কার্যকরী ব্যাচ = ১৬ (৪ × ৪)
    
    # অপটিমাইজেশন (ছোট মডেলের জন্য 3e-4 সবচেয়ে ভালো)
    learning_rate=3e-4,
    warmup_steps=100,
    weight_decay=0.01,
    max_grad_norm=1.0,
    
    # মেমোরি সাশ্রয়
    fp16=False,
    bf16=True,
    gradient_checkpointing=True,
    
    # Logging ও Checkpoint সেভ সেটিংস (প্রতি ৫০০ স্টেপে চেকপয়েন্ট)
    logging_steps=20,
    save_strategy="steps",
    save_steps=500,                        # প্রতি ৫০০ স্টেপে সেভ: checkpoint-500, 1000, 1500
    save_total_limit=4,
    
    # Evaluation
    eval_strategy="steps",
    eval_steps=500,
    
    # SFT সেটিংস
    dataset_text_field="text",
    max_length=1024,
    packing=False,
    remove_unused_columns=False,
    report_to="none",
)

print("✅ Training Arguments সেটআপ সম্পন্ন!")
print(f"\n⏱️ ট্রেনিং পরিকল্পনা:")
print(f"  - মডেল: SmolLM2-135M-Instruct (~135M parameters)")
print(f"  - টার্গেট ডিভাইস: Android 2GB RAM Phones (itel, Symphony)")
print(f"  - কার্যকরী ব্যাচ সাইজ: ১৬ (৪ × ৪)")
print(f"  - মোট স্টেপ: {training_args.max_steps} টি")
print(f"  - চেকপয়েন্ট সেভ হবে: প্রতি ৫০০ স্টেপ পর পর")


## 🚀 ধাপ ৯: Training শুরু করুন!

**এই সেলটি চালালে ট্রেনিং শুরু হবে। এটি ২৫-৪০ মিনিট সময় নিবে।**

আপনি প্রগ্রেস বার দেখে বুঝতে পারবেন কতটা হয়েছে। চা-কফি খেয়ে আসতে পারেন! ☕

In [ ]:
import os
from trl import SFTTrainer
from transformers.trainer_utils import get_last_checkpoint

print("✅ Trainer imports successful!")

# Trainer তৈরি করুন (সকল TRL ও transformers ভার্সনের সাথে সামঞ্জস্যপূর্ণ)
try:
    trainer = SFTTrainer(
        model=model,
        args=training_args,
        train_dataset=formatted_dataset['train'],
        eval_dataset=formatted_dataset['test'],
        processing_class=tokenizer,
    )
except TypeError:
    trainer = SFTTrainer(
        model=model,
        args=training_args,
        train_dataset=formatted_dataset['train'],
        eval_dataset=formatted_dataset['test'],
        tokenizer=tokenizer,
    )

# 🔍 ড্রাইভে আগে থেকে কোনো চেকপয়েন্ট আছে কিনা চেক করা
last_checkpoint = get_last_checkpoint(training_args.output_dir)

if last_checkpoint is not None:
    print(f"🔄 পূর্ববর্তী চেকপয়েন্ট পাওয়া গেছে: {last_checkpoint}")
    print("🚀 ওই চেকপয়েন্ট থেকেই পরবর্তী স্টেপের ট্রেনিং চালু হচ্ছে...\n")
    print("="*60)
    trainer.train(resume_from_checkpoint=last_checkpoint)
else:
    print("🚀 কোনো পুরানো চেকপয়েন্ট নেই, শুরু থেকে (স্টেপ ১) ট্রেনিং শুরু হচ্ছে...\n")
    print("="*60)
    trainer.train()

print("\n" + "="*60)
print("🎉 ট্রেনিং সম্পন্ন! চেকপয়েন্ট ড্রাইভে সেভ হয়েছে...")


## 💾 ধাপ ১০: মডেল সেভ করুন

In [ ]:
final_model_path = "/content/drive/MyDrive/smollm_135m_final"

print(f"💾 মডেল সেভ হচ্ছে: {final_model_path}")

# Adapter এবং Tokenizer সেভ করুন
trainer.model.save_pretrained(final_model_path)
tokenizer.save_pretrained(final_model_path)

print("\n✅ মডেল সফলভাবে Google Drive-এ সেভ হয়েছে!")
print(f"📁 লোকেশন: {final_model_path}")

# সেভ হওয়া ফাইলগুলো দেখুন
!ls -lh {final_model_path}


## 🧪 ধাপ ১১: মডেল টেস্ট করুন

এখন আপনার ট্রেন করা মডেল দিয়ে কিছু প্রশ্নের উত্তর দেখুন!

In [ ]:
from peft import PeftModel

print("🔄 মডেল লোড ও টেস্ট করা হচ্ছে...")

# Base model লোড
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)

# LoRA adapter merge
model_test = PeftModel.from_pretrained(base_model, final_model_path)
model_test = model_test.merge_and_unload()

print("✅ মডেল প্রস্তুত!\n")

def test_model(question):
    prompt = f"""<|im_start|>system
You are a helpful AI assistant for Bengali education.<|im_end|>
<|im_start|>user
{question}<|im_end|>
<|im_start|>assistant
"""
    inputs = tokenizer(prompt, return_tensors="pt").to(model_test.device)
    outputs = model_test.generate(
        **inputs,
        max_new_tokens=256,
        temperature=0.7,
        top_p=0.9,
        repetition_penalty=1.1,
        do_sample=True,
        pad_token_id=tokenizer.pad_token_id,
    )
    response = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
    return response.strip()

# টেস্ট প্রশ্নসমূহ
test_questions = [
    "হাই, কেমন আছো?",
    "বাংলাদেশের রাজধানী কী?",
    "বাংলা বর্ণমালায় স্বরবর্ণ কয়টি?",
]

for q in test_questions:
    print(f"❓ প্রশ্ন: {q}")
    print(f"🤖 উত্তর: {test_model(q)}\n")


In [ ]:
# টেস্ট প্রশ্ন
test_questions = [
    "বাংলাদেশের রাজধানী কী?",
    "সালোকসংশ্লেষণ কাকে বলে?",
    "পিথাগোরাসের উপপাদ্য কী?",
    "বাংলা বর্ণমালায় মোট কয়টি বর্ণ আছে?",
    "What is the capital of Bangladesh?",
]

for i, question in enumerate(test_questions, 1):
    print(f"\n❓ প্রশ্ন {i}: {question}")
    answer = test_model(question)
    print(f"✅ উত্তর: {answer}")
    print("-"*60)

## 🎨 ধাপ ১২: নিজের প্রশ্ন করুন!

এখন আপনি নিজে কোনো প্রশ্ন করে দেখতে পারেন।

In [ ]:
# আপনার প্রশ্ন এখানে লিখুন
my_question = "মানবদেহে কয়টি হাড় আছে?"  # ✏️ এখানে আপনার প্রশ্ন লিখুন

print(f"❓ আপনার প্রশ্ন: {my_question}\n")
answer = test_model(my_question)
print(f"✅ মডেলের উত্তর:\n{answer}")

## 📤 ধাপ ১৩: মডেল ডাউনলোড করুন (Optional)

যদি আপনি মডেল লোকাল মেশিনে ডাউনলোড করতে চান:

In [ ]:
!cd /content/drive/MyDrive && zip -r smollm_135m_final.zip smollm_135m_final/
print("✅ ZIP ফাইল তৈরি সম্পন্ন!")
print("📁 লোকেশন: /content/drive/MyDrive/smollm_135m_final.zip")


## 📊 ধাপ ১৪: Training মেট্রিক্স দেখুন

In [ ]:
# Training history দেখুন
import pandas as pd
import matplotlib.pyplot as plt

# Training logs থেকে loss extract করুন
logs = trainer.state.log_history

train_loss = [log['loss'] for log in logs if 'loss' in log]
eval_loss = [log['eval_loss'] for log in logs if 'eval_loss' in log]

# Plot
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.plot(train_loss, label='Training Loss')
plt.xlabel('Steps')
plt.ylabel('Loss')
plt.title('Training Loss Over Time')
plt.legend()
plt.grid(True)

plt.subplot(1, 2, 2)
if eval_loss:
    plt.plot(eval_loss, label='Eval Loss', color='orange')
    plt.xlabel('Eval Steps')
    plt.ylabel('Loss')
    plt.title('Evaluation Loss Over Time')
    plt.legend()
    plt.grid(True)

plt.tight_layout()
plt.show()

print(f"\n📊 Final Training Loss: {train_loss[-1]:.4f}")
if eval_loss:
    print(f"📊 Final Eval Loss: {eval_loss[-1]:.4f}")

## 🎉 অভিনন্দন!

আপনি সফলভাবে **SmolLM2-135M** (100M Class) মডেলকে বাংলা ডেটাসেট দিয়ে ফাইন-টিউন করেছেন।

### 📱 পরবর্তী ধাপ (GGUF-এ কনভার্ট করে itel ফোনে চালানো):
1. মডেলটি GGUF Q4_K_M ফরম্যাটে কনভার্ট করলে সাইজ হবে মাত্র **~৮৫ MB**।
2. এটি ২ GB RAM মোবাইলে মাত্র **~১৫০ MB RAM** নিবে, ফলে কোনো ক্র্যাশ ছাড়াই সুপারফাস্ট চলবে।